# Phase 2 — Job Embeddings and Top-K Search
This notebook follows the Day 2 flow: inspect source documents → embed only new/changed rows → verify Lakebase vectors → run semantic top-K retrieval.

In [ ]:
%pip install pg8000 databricks-sdk
dbutils.library.restartPython()

## Locate the Databricks Git repository

In [ ]:
import sys
from pathlib import Path

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
workspace_path = Path(notebook_path)
if not str(workspace_path).startswith('/Workspace/'):
    workspace_path = Path('/Workspace') / str(workspace_path).lstrip('/')
REPO_ROOT = workspace_path.parent.parent
if not (REPO_ROOT / 'job_embeddings.py').is_file():
    raise RuntimeError(f'Could not locate job_embeddings.py from {workspace_path}')
sys.path.insert(0, str(REPO_ROOT))
print(f'Repository root: {REPO_ROOT}')

## Import the shared application modules

In [ ]:
import config
import lakebase
from embedding_model import embed_query
from job_embeddings import embedding_status, embed_pending_postings, fetch_pending_postings, semantic_search

print(f'Embedding endpoint: {config.EMBEDDING_MODEL}')
print(f'Expected dimensions: {config.EMBEDDING_DIMENSION}')
print(lakebase.ping())

## Verify the hosted model before writing data

In [ ]:
probe = embed_query('senior data engineer with Databricks, Python, SQL, and AWS')
assert len(probe) == 1024, f'Expected 1024 values, got {len(probe)}'
print(f'Model returned {len(probe)} dimensions')

## Inspect pending postings

In [ ]:
print('Before:', embedding_status())
display(fetch_pending_postings(limit=20))

## Embed new or content-changed postings

In [ ]:
result = embed_pending_postings(limit=100, batch_size=8)
print(result)

## Verify persisted chunks

In [ ]:
display(lakebase.run_query('''
SELECT job_posting_id, count(*) AS chunks, max(model_name) AS model_name
FROM job_posting_embeddings
GROUP BY job_posting_id
ORDER BY chunks DESC
LIMIT 20
'''))

## Top-K semantic search

In [ ]:
TOP_K = 5
QUERY = 'senior data engineer with Databricks, Python, SQL, and AWS'
matches = semantic_search(QUERY, top_k=TOP_K)
display(matches)

## Filtered top-K example

In [ ]:
remote_matches = semantic_search(
    'machine learning platform engineer',
    top_k=10,
    remote_only=True,
    sources=['adzuna', 'remoteok'],
)
display(remote_matches)

## Idempotency check
Run `embed_pending_postings` again. Unchanged postings should not be selected. A later job sync only re-embeds postings whose `content_hash` changed.

In [ ]:
print(embed_pending_postings(limit=100, batch_size=8))